In [ ]:
import pandas as pd
import numpy as np
import pyemu
import yaml
import math
import os
from scipy.special import exp1
import plotly.graph_objects as go

In [ ]:
import os, shutil, stat
from pathlib import Path

def on_rm_error(func, path, excinfo):
    # try to change file permission and retry
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print("on_rm_error could not remove:", path, type(e), e)

template_path = Path(r"C:\Python\Personal\proj6\codes\runs\mar")
os.chdir(Path.home())

print("Attempting robust rmtree on:", template_path)
try:
    shutil.rmtree(template_path, onerror=on_rm_error)
    print("Removed", template_path)
except Exception as e:
    print("Still failed:", type(e), e)

In [ ]:
# set up path
target_dir = r"C:\Python\Personal\proj6\codes\runs\mar"
rel_t_d = os.path.relpath(target_dir, os.getcwd()) 
os.chdir(rel_t_d)

In [51]:
import os
import sys
import numpy as np
import pandas as pd
import yaml
import math
from scipy.special import exp1
import pyemu

##########################
### pit-centric domain ###
##########################
n_wells = 2
num_reals = 150
x_min, x_max = -50, 50
y_min, y_max = -50, 50
no_go_radius = 10.0
q_min, q_max = 6.5, 65
T_val, S_val, t_eval_val = 100.0, 0.001, 365.0

# Paths
mou_exe = r"C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe"
model_script = r"C:\Python\Personal\proj6\codes\scripts\run_theis_fwd_dd.py"
python_exe = sys.executable

######################
### parameter list ###
######################
par_names = []
for i in range(1, n_wells + 1):
    par_names.extend([f"well_{i}_x", f"well_{i}_y", f"well_{i}_q"])
for p in ["xc", "yc"]:
    if p not in par_names:
        par_names.append(p)

#######################
### initialize pest ###
#######################
pst = pyemu.pst_utils.generic_pst(par_names=par_names)
pdf = pst.parameter_data

# Randomize compliance point y-position once
yc_fixed = np.random.choice([y_min, y_max])

# Configure metadata
well_pars = [p for p in par_names if "well" in p]
pdf.loc[well_pars, ["partrans", "parchglim", "pargp"]] = ["none", "factor", "well_pos"]
pdf.loc[well_pars, ["parlbnd", "parubnd"]] = [x_min, x_max]

well_q_pars = [p for p in par_names if "_q" in p]
pdf.loc[well_q_pars, ["partrans", "parchglim", "pargp"]] = ["none", "factor", "pumping_rate"]
pdf.loc[well_q_pars, "parlbnd"] = q_min
pdf.loc[well_q_pars, "parubnd"] = q_max

pdf.loc["xc", ["partrans", "parchglim", "pargp"]] = ["none", "factor", "compliance"]
pdf.loc["xc", ["parlbnd", "parubnd"]] = [x_min, x_max]

# Fix yc as a parameter so it appears in config.yml
pdf.loc["yc", ["parval1", "parlbnd", "parubnd", "pargp", "partrans"]] = [yc_fixed, yc_fixed, yc_fixed, "compliance", "fixed"]

###################
### random draw ###
###################
valid_data = []
while len(valid_data) < num_reals:
    row = np.random.uniform(0, 1, len(par_names))
    scaled_row = [pdf.loc[p, "parlbnd"] + r * (pdf.loc[p, "parubnd"] - pdf.loc[p, "parlbnd"]) for p, r in zip(par_names, row)]
    well_coords = [[scaled_row[par_names.index(f"well_{i}_x")], scaled_row[par_names.index(f"well_{i}_y")]] for i in range(1, n_wells + 1)]
    if np.all(np.linalg.norm(well_coords, axis=1) > no_go_radius):
        valid_data.append(scaled_row)

pop = pyemu.ParameterEnsemble(pst=pst, df=pd.DataFrame(valid_data, columns=par_names))
pst.parameter_data.loc[par_names, "parval1"] = pop.iloc[0, :].values

###########################
### objective functions ###
###########################
template_row = pst.observation_data.loc["obs1", :].copy()
pst.observation_data = pst.observation_data.iloc[0:0] 
for o in ["obj_min_xc", "obj_min_q"]:
    pst.observation_data.loc[o, :] = template_row
pst.observation_data.loc["obj_min_xc", ["obsval", "weight", "obgnme"]] = [0.0, 1.0, "min_xc_gp"]
pst.observation_data.loc["obj_min_q", ["obsval", "weight", "obgnme"]] = [0.0, 0.1, "min_q_gp"]
pst.observation_data["obsnme"] = pst.observation_data.index

###########################
### MOU solver settings ###
###########################
pst.pestpp_options["mou_objectives"] = "min_xc_gp,min_q_gp"
pst.pestpp_options["mou_obj_grp_sense"] = "min,min"
pst.pestpp_options["mou_population_size"] = num_reals
pst.pestpp_options["mou_generator"] = "de"
pst.pestpp_options["mou_max_generations"] = 50
pst.pestpp_options["base_ensemble"] = "pop.csv"
pst.control_data.noptmax = 10

########################
### write and record ###
########################
pst_filename = "mou_model.pst"

# 1. Map Files
pst.model_input_data = pd.DataFrame({"pest_file": ["mou_model.tpl"], "model_file": ["config.yml"]})
pst.model_output_data = pd.DataFrame({"pest_file": ["mou_model.ins"], "model_file": ["allobs.out"]})
pst.model_command = "model.bat"

# 2. Create Supporting Files (Ultra-Safe Formatting)
# Template: 20 spaces between tildes ensures NO overflow
with open("mou_model.tpl", "w", newline="\n") as f:
    f.write("ptf ~\n")
    f.write("T: 100.0\nS: 0.001\nt_eval: 365.0\n")
    f.write("xc: ~                   xc                   ~\n")
    f.write("yc: ~                   yc                   ~\n")
    f.write("well_1:\n  x: ~                   well_1_x                   ~\n")
    f.write("  y: ~                   well_1_y                   ~\n")
    f.write("  Q: ~                   well_1_q                   ~\n")
    f.write("well_2:\n  x: ~                   well_2_x                   ~\n")
    f.write("  y: ~                   well_2_y                   ~\n")
    f.write("  Q: ~                   well_2_q                   ~\n")

# Instruction: Standard fixed-string marker + capture bracket
with open("mou_model.ins", "w", newline="\n") as f:
    f.write("pif ~\n")
    f.write("l1 !res_xc! [obj_min_xc]\n")
    f.write("l1 !res_q!  [obj_min_q]\n")

# Dummy Output: Must match the .ins labels EXACTLY
with open("allobs.out", "w", newline="\n") as f:
    f.write("res_xc 0.0\n")
    f.write("res_q  0.0\n")

# Batch File: Ensure quotes around paths
with open("model.bat", "w", newline="\n") as f:
    f.write(f"@echo off\n\"{python_exe}\" \"{model_script}\"\n")

# 3. Write PST & Append Options
# safe write + append of pestpp options (replace the old temp_opts / clear block)
temp_opts = pst.pestpp_options.copy()
pst.write(pst_filename, version=1)

with open(pst_filename, "a", newline="\n") as f:
    f.write("\n* pestpp options\n")
    f.write("++forgive_unknown_args(1)\n")
    # ensure mou_objectives is serialized cleanly if present
    if "mou_objectives" in temp_opts:
        mo = temp_opts.pop("mou_objectives")
        mo_str = ",".join(map(str, mo)) if isinstance(mo, (list, tuple)) else str(mo)
        f.write(f"++mou_objectives({mo_str})\n")
    for k, v in temp_opts.items():
        if "objective" in k.lower():
            continue
        v_str = ",".join(map(str, v)) if isinstance(v, (list, tuple)) else str(v)
        f.write(f"++{k}({v_str})\n")

# 4. Save Ensemble (No header, No index)
par_order = pst.parameter_data.index.tolist()
pop.loc[:, par_order].to_csv("pop.csv", index=False, header=False, lineterminator='\n')

print("Launching PESTPP-MOU 5.2.25...")
# Remove any quotes from mou_exe path for the run call
clean_mou_exe = mou_exe.replace('"', '')
pyemu.os_utils.run(f"{clean_mou_exe} test.pst", use_sp=True)


noptmax:10, npar_adj:7, nnz_obs:2
Launching PESTPP-MOU 5.2.25...


Exception: run() returned non-zero: 3221226505

### plotting

In [ ]:
##############
### set up ###
##############

fig = go.Figure()

# access the underlying dataframe using the internal _df attribute
df = pop._df

######################################
### two well example visualization ###
######################################

# domain
fig.add_shape(type="rect", x0=x_min, y0=y_min, x1=x_max, y1=y_max, line=dict(color="black"), opacity=0.1)

# no-go area
fig.add_shape(type="circle", x0=-no_go_radius, y0=-no_go_radius, x1=no_go_radius, y1=no_go_radius, line=dict(color="black", dash="dash"))

# targets with drawdown labels
fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers+text', text=[f"Open Pit<br>s: {s_pit:.2f}m"],
                         marker=dict(color='black', size=10, symbol='hexagon'), name="Open Pit", textposition="top center"))

fig.add_trace(go.Scatter(x=[xc_val], y=[Yc], mode='markers+text', text=[f"Compliance Point<br>s: {s_compliance:.2f}m"],
                         marker=dict(color='yellow', size=10, symbol='star', line=dict(width=1, color='black')), 
                         name="Compliance Point", textposition="top center" if Yc == y_min else "bottom center"))

# well ensembles
fig.add_trace(go.Scatter(x=df["well_1_x"], y=df["well_1_y"], mode='markers', marker=dict(color='lightskyblue', opacity=0.2), name="Well 1 Ensemble"))
fig.add_trace(go.Scatter(x=df["well_2_x"], y=df["well_2_y"], mode='markers', marker=dict(color='lightcoral', opacity=0.2), name="Well 2 Ensemble"))

# selected wells
fig.add_trace(go.Scatter(x=[pst.parameter_data.loc["well_1_x", "parval1"]], y=[pst.parameter_data.loc["well_1_y", "parval1"]],
                         mode='markers+text', text=["Well 1"], marker=dict(color='dodgerblue', size=10), name="Well 1 Selected", textposition="top center"))
fig.add_trace(go.Scatter(x=[pst.parameter_data.loc["well_2_x", "parval1"]], y=[pst.parameter_data.loc["well_2_y", "parval1"]],
                         mode='markers+text', text=["Well 2"], marker=dict(color='crimson', size=10, symbol='square'), name="Well 2 Selected", textposition="top center"))

# CoM and distance lines
fig.add_trace(go.Scatter(x=[com_x], y=[com_y], mode='markers+text', text=["CoM"], marker=dict(color='black', size=12, symbol='x'), name="Pumping Center of Mass", textposition="top center"))
fig.add_trace(go.Scatter(x=[com_x, xc_val, None, com_x, 0], y=[com_y, Yc, None, com_y, 0], 
                         mode='lines', line=dict(color='grey', width=1, dash='dot'), showlegend=False))

fig.update_layout(
    xaxis_title="X Coordinate", 
    yaxis_title="Y Coordinate", 
    template="plotly_white", 
    width=700, 
    height=700,
    title=(f"Theis Analysis: Realization {selected_real}<br>"
           f"<sup>Aquifer Parameters: T={T_val} m²/d, S={S_val}</sup>")
)

fig.show()

NameError: name 's_pit' is not defined

In [ ]:
# !{mou_exe} mou_model.pst
!{mou_exe} --help